# Verify Shawn's data handoff — Zichen Luo
Run after `01_data_cleaning.ipynb` produces `zichen_training_data.zip`. This notebook checks the bundle and loads the training/validation/test manifests; it does **not** fit a model. Keep the model splits and class numbering fixed.

Place Shawn's ZIP in `/MyDrive/DS4002_Project3/input/` and use a CPU runtime for verification.

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import zipfile
import hashlib
from PIL import Image, ImageOps

drive.mount('/content/drive')
BUNDLE = Path('/content/drive/MyDrive/DS4002_Project3/input/zichen_training_data.zip')
DEST = Path('/content/project3_training')
DEST.mkdir(exist_ok=True)
# Only extract safe relative paths inside the target directory.
with zipfile.ZipFile(BUNDLE) as z:
    for info in z.infolist():
        target = (DEST / info.filename).resolve()
        if not target.is_relative_to(DEST.resolve()):
            raise ValueError('Unsafe archive path')
    z.extractall(DEST)
manifest = pd.read_csv(DEST/'DATA/clean_manifest.csv', dtype={'Adv_ID':str,'Genmodel_ID':str,'Image_ID':str})
assert set(manifest.Split)=={'train','validation','test'}
assert manifest.groupby('Genmodel_ID').Split.nunique().eq(1).all()
assert manifest.groupby('Adv_ID').Split.nunique().eq(1).all()
assert manifest.groupby('Adv_ID').size().eq(3).all()
assert manifest.groupby('Adv_ID').View.nunique().eq(3).all()
assert manifest.Pixel_sha256.is_unique
assert manifest.Image_ID.is_unique
for split in ['train','validation','test']:
    rows = manifest[manifest.Split.eq(split)]
    assert set(rows.Body_type)=={'Sedan','SUV','Pickup'}
    assert set(rows.View)=={'Front','Side','Rear'}
for row in manifest.itertuples(index=False):
    p=DEST/row.Path
    assert p.is_file(), str(p)
    with Image.open(p) as im:
        im=ImageOps.exif_transpose(im).convert('RGB'); im.load()
        digest=hashlib.sha256(f'{im.size}'.encode()+im.tobytes()).hexdigest()
        assert digest==row.Pixel_sha256, str(p)
print('PASS: readable files, matching hashes, matched triplets, no model overlap, all classes in each split.')
display(manifest.groupby(['Split','Body_type']).agg(Images=('Image_ID','size'),Listings=('Adv_ID','nunique'),Models=('Genmodel_ID','nunique')))

In [ ]:
# These fixed tables are the input contract for your modeling notebook.
train_df = manifest[manifest.Split.eq('train')].copy()
validation_df = manifest[manifest.Split.eq('validation')].copy()
test_df = manifest[manifest.Split.eq('test')].copy()
CLASS_TO_ID = {'Sedan':0, 'SUV':1, 'Pickup':2}
assert manifest.Class_id.eq(manifest.Body_type.map(CLASS_TO_ID)).all()
# Load photos using Image.open(DEST / row.Path).convert('RGB').
# Supply Path and Class_id to your PyTorch Dataset; do not train on filenames/IDs.
print('Class mapping:', CLASS_TO_ID)

## Modeling work to add
Fine-tune ImageNet-pretrained ResNet-18 with training-only augmentation. Derive class weights from `train_df` only if needed. Use validation macro F1 for tuning/early stopping, then freeze choices before testing. Do not run five-fold cross-validation in addition to the fixed split.

Evaluate the **same test listings** at each viewpoint. Save predictions containing `Adv_ID`, `Genmodel_ID`, `View`, true class, predicted class, and probabilities. Report balanced accuracy, macro F1, macro precision, per-class recall, and normalized confusion matrices by view. The finish line is balanced accuracy > 1/3 for all three views, whether or not achieved. Keep results honest and report the pickup test model count. This observational comparison does not establish a causal effect of angle.

Add the completed training/evaluation notebook to `SCRIPTS/` and its actual outputs to `OUTPUT/`; update README with exact run instructions and package versions.